# Какие тесты нужны — план покрытия

Фреймворк: **Google Test (gtest)**, язык **C++17**

Все тесты пишутся в папке `tests/` и компилируются отдельно от GUI.

Синтаксис:
```cpp
TEST(ГруппаТестов, НазваниеТеста) {
    // arrange — подготовка данных
    // act     — вызов тестируемой функции
    // assert  — проверка результата
    EXPECT_EQ(ожидаемое, фактическое);
}
```

---
## 1. Тесты парсера пакетов
📄 Файл: `tests/test_parser.cpp`

### ✅ Валидный пакет корректно парсится
```cpp
TEST(ParserTest, ValidPacketParsedCorrectly) {
    auto pkt = makePacket(0x01, {0x00, 0x00, 0x80, 0x3F, 0x64, 0x00});

    ParsedPacket result = parsePacket(pkt.data(), pkt.size());

    EXPECT_TRUE(result.isValid);
}
```

### ✅ Пакет с неверным CRC отклоняется
```cpp
TEST(ParserTest, InvalidCRCRejected) {
    auto pkt = makePacket(0x01, {0x00, 0x00, 0x80, 0x3F}, /*corruptCRC=*/true);

    ParsedPacket result = parsePacket(pkt.data(), pkt.size());

    EXPECT_FALSE(result.isValid);
}
```

### ✅ Неверный SYNC пропускается, поиск продолжается
```cpp
TEST(ParserTest, WrongSync1Rejected) {
    auto pkt = makePacket(0x01, {0x01, 0x02});
    pkt[0] = 0xBB; // портим SYNC1: должен быть 0xAA

    ParsedPacket result = parsePacket(pkt.data(), pkt.size());

    EXPECT_FALSE(result.isValid);
}

TEST(ParserTest, WrongSync2Rejected) {
    auto pkt = makePacket(0x01, {0x01, 0x02});
    pkt[1] = 0x66; // портим SYNC2: должен быть 0x55

    ParsedPacket result = parsePacket(pkt.data(), pkt.size());

    EXPECT_FALSE(result.isValid);
}
```

### ✅ Неполный пакет (обрыв данных) не вызывает краш
```cpp
TEST(ParserTest, IncompletePacketNoCrash) {
    auto pkt = makePacket(0x01, {0x01, 0x02, 0x03, 0x04});
    pkt.resize(pkt.size() - 3); // обрезаем конец — имитируем обрыв

    // не должен упасть с segfault или исключением
    ParsedPacket result = parsePacket(pkt.data(), pkt.size());

    EXPECT_FALSE(result.isValid);
}
```

### ✅ Парсер правильно извлекает дистанцию из MEASUREMENT
```cpp
TEST(ParserTest, MeasurementDistanceDecoded) {
    // float 1.5м в little-endian: 0x00 0x00 0xC0 0x3F
    auto pkt = makePacket(0x01, {0x00, 0x00, 0xC0, 0x3F, 0x00, 0x00});

    ParsedPacket result = parsePacket(pkt.data(), pkt.size());

    EXPECT_TRUE(result.isValid);
    EXPECT_NEAR(result.measurement.distance_m, 1.5f, 0.001f);
}
```

### ✅ Парсер правильно извлекает интенсивность (magnitude)
```cpp
TEST(ParserTest, MeasurementMagnitudeDecoded) {
    // mag = 300 → 0x2C 0x01 (little-endian uint16)
    auto pkt = makePacket(0x01, {0x00, 0x00, 0x80, 0x3F, 0x2C, 0x01});

    ParsedPacket result = parsePacket(pkt.data(), pkt.size());

    EXPECT_TRUE(result.isValid);
    EXPECT_EQ(result.measurement.magnitude, 300);
}
```

### ✅ Парсер обрабатывает INFO пакет (ASCII)
```cpp
TEST(ParserTest, InfoPacketTextExtracted) {
    // payload: ASCII строка "OK"
    auto pkt = makePacket(0x02, {0x4F, 0x4B});

    ParsedPacket result = parsePacket(pkt.data(), pkt.size());

    EXPECT_TRUE(result.isValid);
    EXPECT_EQ(result.type, MessageType::INFO);
    EXPECT_EQ(result.infoText, "OK");
}
```

### ✅ Парсер обрабатывает STATUS пакет
```cpp
TEST(ParserTest, StatusPacketDecoded) {
    auto pkt = makePacket(0x03, {0x00}); // статус 0x00

    ParsedPacket result = parsePacket(pkt.data(), pkt.size());

    EXPECT_TRUE(result.isValid);
    EXPECT_EQ(result.type, MessageType::STATUS);
    EXPECT_EQ(result.statusCode, 0x00);
}
```

### ✅ ACK и NACK корректно различаются
```cpp
TEST(ParserTest, AckDistinguishedFromNack) {
    auto ackPkt  = makePacket(0x04, {});
    auto nackPkt = makePacket(0x05, {});

    ParsedPacket ack  = parsePacket(ackPkt.data(),  ackPkt.size());
    ParsedPacket nack = parsePacket(nackPkt.data(), nackPkt.size());

    EXPECT_EQ(ack.type,  MessageType::ACK);
    EXPECT_EQ(nack.type, MessageType::NACK);
    EXPECT_TRUE(ack.isAck);
    EXPECT_FALSE(nack.isAck);
}
```

### ✅ Несколько пакетов подряд в буфере парсятся все
```cpp
TEST(ParserStreamTest, MultiplePacketsParsedSequentially) {
    auto pkt1 = makePacket(0x01, {0x00, 0x00, 0x80, 0x3F, 0x01, 0x00});
    auto pkt2 = makePacket(0x03, {0x00});

    std::vector<uint8_t> stream;
    stream.insert(stream.end(), pkt1.begin(), pkt1.end());
    stream.insert(stream.end(), pkt2.begin(), pkt2.end());

    size_t consumed = 0;
    ParsedPacket r1 = findNextPacket(stream.data(), stream.size(), consumed);
    ParsedPacket r2 = findNextPacket(stream.data() + consumed,
                                     stream.size() - consumed, consumed);

    EXPECT_TRUE(r1.isValid);
    EXPECT_TRUE(r2.isValid);
    EXPECT_EQ(r1.type, MessageType::MEASUREMENT);
    EXPECT_EQ(r2.type, MessageType::STATUS);
}
```

### ✅ «Мусор» между пакетами не ломает парсинг
```cpp
TEST(ParserStreamTest, GarbageBetweenPackets) {
    std::vector<uint8_t> stream = {0x00, 0xFF, 0x12}; // мусор
    auto pkt = makePacket(0x01, {0x00, 0x00, 0x80, 0x3F, 0x64, 0x00});
    stream.insert(stream.end(), pkt.begin(), pkt.end());

    size_t consumed = 0;
    ParsedPacket result = findNextPacket(stream.data(), stream.size(), consumed);

    EXPECT_TRUE(result.isValid);
    EXPECT_EQ(consumed, 3 + pkt.size()); // пропустил 3 байта мусора
}
```

### ✅ Длинный мусорный поток не зависает и не съедает память
```cpp
TEST(ParserStreamTest, LongGarbageStreamNoCrash) {
    std::vector<uint8_t> garbage(10000, 0xFF); // 10к байт мусора

    size_t consumed = 0;
    ParsedPacket result = findNextPacket(garbage.data(), garbage.size(), consumed);

    // не нашёл пакет — это нормально, главное не завис
    EXPECT_FALSE(result.isValid);
}
```

---
## 2. Тесты CRC16 (основные на первых этапах)
📄 Файл: `tests/test_crc.cpp`

### ✅ CRC16 считается правильно (по эталонным векторам)
```cpp
TEST(CRC16Test, KnownVector) {
    uint8_t data[] = {0x01, 0x02, 0x03};

    uint16_t result = calculateCRC16(data, 3);

    // TODO: подставить эталонное значение после того
    // как коллега уточнит алгоритм (полином и начальное значение)
    EXPECT_EQ(result, 0xXXXX);
}
```

### ✅ Два одинаковых вызова дают одинаковый результат
```cpp
TEST(CRC16Test, Deterministic) {
    uint8_t data[] = {0xAA, 0x55, 0x01, 0x04};

    EXPECT_EQ(calculateCRC16(data, 4), calculateCRC16(data, 4));
}
```

### ✅ Изменение одного байта меняет CRC
```cpp
TEST(CRC16Test, BitFlipChangesCRC) {
    uint8_t original[] = {0x10, 0x20, 0x30, 0x40};
    uint8_t corrupted[] = {0x10, 0x20, 0x30, 0x41}; // последний бит изменён

    EXPECT_NE(calculateCRC16(original, 4), calculateCRC16(corrupted, 4));
}
```

---
## 3. Тесты формирования команд (TX)
📄 Файл: `tests/test_commands.cpp`

### ✅ Команда START имеет правильный заголовок и CRC
```cpp
TEST(CommandsTest, StartCommandHasValidStructure) {
    auto cmd = buildStartCommand();

    // проверяем SYNC байты
    EXPECT_EQ(cmd[0], 0xAA);
    EXPECT_EQ(cmd[1], 0x55);
    // проверяем что CRC в конце совпадает с вычисленным
    assertValidCommandStructure(cmd); // вспомогательная функция
}
```

### ✅ Команда STOP корректно формируется
```cpp
TEST(CommandsTest, StopCommandHasValidStructure) {
    auto cmd = buildStopCommand();

    assertValidCommandStructure(cmd);
}
```

### ✅ Команда смены режима (SINGLE / CONTINUOUS / DEBUG) корректна
```cpp
TEST(CommandsTest, SetModeSingleShotHasValidStructure) {
    auto cmd = buildSetModeCommand(SensorMode::SINGLE_SHOT);
    assertValidCommandStructure(cmd);
}

TEST(CommandsTest, SetModeContinuousHasValidStructure) {
    auto cmd = buildSetModeCommand(SensorMode::CONTINUOUS);
    assertValidCommandStructure(cmd);
}

TEST(CommandsTest, DifferentModesProduceDifferentCommands) {
    auto single = buildSetModeCommand(SensorMode::SINGLE_SHOT);
    auto cont   = buildSetModeCommand(SensorMode::CONTINUOUS);
    auto debug  = buildSetModeCommand(SensorMode::DEBUG_RAW);

    EXPECT_NE(single, cont);
    EXPECT_NE(cont, debug);
}
```

### ✅ Команда запроса статуса корректна
```cpp
TEST(CommandsTest, StatusRequestHasValidStructure) {
    auto cmd = buildStatusRequestCommand();

    assertValidCommandStructure(cmd);
}
```

### ✅ START и STOP отличаются друг от друга
```cpp
TEST(CommandsTest, StartAndStopAreDifferent) {
    auto start = buildStartCommand();
    auto stop  = buildStopCommand();

    EXPECT_NE(start, stop);
}
```

---
## 4. Тесты логирования в CSV
📄 Файл: `tests/test_logger.cpp`

### ✅ CSV создаётся с правильными заголовками
```cpp
TEST(CSVLoggerTest, CSVHeaderCreated) {
    CSVLogger logger("test_output.csv");
    logger.open();
    logger.close();

    std::ifstream f("test_output.csv");
    std::string header;
    std::getline(f, header);

    EXPECT_EQ(header, "timestamp,distance_m,magnitude,phase");
}
```

### ✅ Строка с измерением корректно записывается
```cpp
TEST(CSVLoggerTest, MeasurementRowWritten) {
    CSVLogger logger("test_output.csv");
    logger.open();
    logger.write({1.5f, 300, 0.0f}); // дистанция, интенсивность, фаза
    logger.close();

    std::ifstream f("test_output.csv");
    std::string header, row;
    std::getline(f, header);
    std::getline(f, row);

    EXPECT_NE(row.find("1.5"), std::string::npos);
    EXPECT_NE(row.find("300"), std::string::npos);
}
```

### ✅ Несколько записей не перезаписывают друг друга
```cpp
TEST(CSVLoggerTest, MultipleRowsAppended) {
    CSVLogger logger("test_output.csv");
    logger.open();
    logger.write({1.0f, 100, 0.0f});
    logger.write({2.0f, 200, 0.0f});
    logger.write({3.0f, 300, 0.0f});
    logger.close();

    // считаем строки в файле: 1 заголовок + 3 строки данных = 4
    std::ifstream f("test_output.csv");
    int lineCount = std::count(std::istreambuf_iterator<char>(f),
                               std::istreambuf_iterator<char>(), '\n');
    EXPECT_EQ(lineCount, 4);
}
```

### ✅ Файл корректно закрывается после остановки
```cpp
TEST(CSVLoggerTest, FileClosedOnStop) {
    {
        CSVLogger logger("test_output.csv");
        logger.open();
        logger.write({1.0f, 100, 0.0f});
        logger.close(); // явный close
    } // или деструктор — файл должен быть читаем сразу после

    std::ifstream f("test_output.csv");
    EXPECT_TRUE(f.good()); // файл открывается и не повреждён
}
```

### ✅ Ошибка записи (нет прав / диск заполнен) не вызывает краш
```cpp
TEST(CSVLoggerTest, WriteToReadOnlyPathNoCrash) {
    // путь куда заведомо нет прав на запись
    CSVLogger logger("C:\\Windows\\System32\\test.csv");

    // не должен бросить необработанное исключение или упасть
    EXPECT_NO_THROW(logger.open());

    // логгер должен сообщить об ошибке через isOpen()
    EXPECT_FALSE(logger.isOpen());
}
```

---
## 5. Тесты COM-порта (без реального железа)
📄 Файл: `tests/test_serial.cpp`

> Используем **mock-объект** — фиктивный COM-порт, который возвращает заранее заданные байты вместо реального железа.

### ✅ При недоступном порте — понятная ошибка, не краш
```cpp
TEST(SerialPortTest, UnavailablePortErrorHandled) {
    SerialPort port("COM99", 115200); // заведомо несуществующий порт

    bool result = port.open();

    EXPECT_FALSE(result);           // вернул false, не упал
    EXPECT_FALSE(port.isOpen());
    EXPECT_FALSE(port.getLastError().empty()); // есть сообщение об ошибке
}
```

### ✅ Переподключение после обрыва работает
```cpp
TEST(SerialPortTest, ReconnectAfterDisconnect) {
    MockSerialPort port;
    port.open();
    port.simulateDisconnect(); // имитируем обрыв кабеля

    EXPECT_FALSE(port.isOpen());

    bool reconnected = port.open(); // пробуем переподключиться

    EXPECT_TRUE(reconnected);
    EXPECT_TRUE(port.isOpen());
}
```

### ✅ Чтение из mock-порта запускает парсер
```cpp
TEST(SerialPortTest, ReadTriggerParser) {
    MockSerialPort port;
    auto pkt = makePacket(0x01, {0x00, 0x00, 0x80, 0x3F, 0x64, 0x00});
    port.injectBytes(pkt); // «вливаем» байты как будто пришли от сенсора

    port.open();
    auto received = port.readBytes(pkt.size());
    ParsedPacket result = parsePacket(received.data(), received.size());

    EXPECT_TRUE(result.isValid);
    EXPECT_EQ(result.type, MessageType::MEASUREMENT);
}
```

### ✅ Таймаут при отсутствии данных от сенсора
```cpp
TEST(SerialPortTest, TimeoutWhenNoData) {
    MockSerialPort port;
    port.open();
    // ничего не вливаем — порт молчит

    auto start = std::chrono::steady_clock::now();
    auto data  = port.readBytes(10); // пробуем прочитать 10 байт
    auto elapsed = std::chrono::steady_clock::now() - start;

    EXPECT_TRUE(data.empty());  // данных нет
    // таймаут сработал не позже чем через 1 секунду
    EXPECT_LT(elapsed, std::chrono::seconds(1));
}
```

### ✅ Буфер переполнен — старые данные не теряются / нет краша
```cpp
TEST(SerialPortTest, BufferOverflowNoCrash) {
    MockSerialPort port;
    // вливаем больше данных чем вмещает внутренний буфер
    std::vector<uint8_t> flood(100000, 0xAA);
    port.injectBytes(flood);

    port.open();

    // не должен упасть, не должен зависнуть
    EXPECT_NO_THROW(port.readBytes(1024));
}
```